# QuantFin Exeter — Optimal execution with regime-aware RL

**Team:** Kirill Papka, Thomas Nguyen, Harrison Maxwell, Maksim Kitikov · **Group 18** · University of Exeter

**Executive summary**
- **Problem:** Large orders face a trade-off between market impact and inventory risk.
- **Method:** HMM (or vol-threshold fallback) regimes feed a Gymnasium execution env; PPO/SAC via Stable-Baselines3.
- **Result:** Compare RL vs TWAP, VWAP, and Almgren–Chriss on implementation shortfall (bps).
- **Governance:** Claude-powered explanations with deterministic on-disk cache (`data/cached_llm/`).

## Problem statement

We study whether RL can learn **adaptive** liquidation schedules when conditioned on a **detected market regime**, versus classical benchmarks. See Stage 1 PDF for full stochastic-control notation.

In [10]:
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

ROOT = Path.cwd().resolve()
if (ROOT / "src").is_dir():
    pass
elif (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

from src.data_pipeline import load_split
from src.regime_detector import RegimeDetector, regime_sanity_summary
from src.utils import regime_display_name

TICKER = "SPY"
train = load_split("train", ticker=TICKER)
test = load_split("test", ticker=TICKER)
train.head()

,Close,Volume,realised_vol_20,sigma_daily,amihud_illiquidity,bid_ask_proxy,volume_to_spread,vix_aligned,order_imbalance_daily,n_bbo_bars,spread_bps_mean
date,,,,,,,,,,,
2018-01-30,281.76001,131796419.0,0.087302,0.005500,1.840227e-15,0.012479,1.056173e+10,14.79,0.0,0,NaN
2018-01-31,281.89999,118948131.0,0.087142,0.005489,1.847633e-15,0.009294,1.279826e+10,13.54,0.0,0,NaN
2018-02-01,281.57999,90102470.0,0.086703,0.005462,1.740607e-15,0.008452,1.066005e+10,13.47,0.0,0,NaN
2018-02-02,275.45001,173174790.0,0.121046,0.007625,1.875299e-15,0.017499,9.896446e+09,17.31,0.0,0,NaN
2018-02-05,263.92999,294681816.0,0.193851,0.012211,2.004531e-15,0.047513,6.202176e+09,37.32,0.0,0,NaN


## Data & features

Panels are loaded from `CFADATA/features` (or `CFA_DATA_ROOT`). Canonical columns include `Close`, `realised_vol_20`, `sigma_daily`, Amihud liquidity, and volume.

In [11]:
fig = px.line(train.reset_index(), x="date", y="Close", title=f"{TICKER} close (train)")
fig.show()
fig2 = px.line(train.reset_index(), x="date", y=["realised_vol_20"], title="Annualised realised vol (20d)")
fig2.show()

## Regime detection

Fit on **train** only; overlay regimes on price (key judge chart — also in Streamlit app).

In [12]:
det = RegimeDetector(n_components=2, fallback_threshold=0.24)
det.fit(train)
train_r = train.copy()
train_r["regime"] = det.predict(train)
display(regime_sanity_summary(train_r, train_r["regime"].to_numpy()))

from pathlib import Path
png = ROOT / "notebooks" / "regimes_train.png"
det.plot_regimes(train_r["Close"], train_r["regime"].to_numpy(), str(png))
print("Saved", png)

,count,mean_vol,freq
regime,,,
0,671,0.108556,0.541129
1,569,0.266493,0.458871


Saved /Users/kirik/Desktop/Pythom/CFA/CFATeam18/quantfin_exeter/notebooks/regimes_train.png


## Trading environment (one episode sketch)

Attach regimes to the split used for simulation, then step with a random policy for illustration.

In [13]:
from src.trading_env import OptimalExecutionEnv

sim = train_r.dropna(subset=["Close", "sigma_daily"]).copy()
env = OptimalExecutionEnv(sim, T=10, resample=True, seed=SEED)
obs, _ = env.reset(seed=SEED)
rows = []
for step in range(10):
    a = np.array([0.3], dtype=np.float32)
    obs, r, term, trunc, info = env.step(a)
    rows.append({"step": step, "reward": r, **{k: info[k] for k in ("execution_cost", "inventory_risk", "regime")}})
    if term or trunc:
        break
pd.DataFrame(rows)

,step,reward,execution_cost,inventory_risk,regime
0,0,-0.000915,0.000900,3.024498e-05,0
1,1,-0.000512,0.000504,1.673038e-05,0
2,2,-0.000295,0.000291,8.338352e-06,0
3,3,-0.000176,0.000173,4.392631e-06,0
4,4,-0.000108,0.000107,2.373365e-06,0
5,5,-0.000068,0.000067,1.155482e-06,0
6,6,-0.000044,0.000044,5.406734e-07,0
7,7,-0.000029,0.000029,2.668429e-07,0
8,8,-0.000019,0.000019,1.271783e-07,0
9,9,-0.009588,0.000013,6.410213e-08,0


## RL training & benchmarks

Run `python scripts/train.py --train --timesteps 200000` for full training; tensorboard logs in `logs/`. Below is a placeholder cell — uncomment when SB3 is installed.

In [ ]:
# from stable_baselines3 import PPO
# from src.rl_agent import train_agent, evaluate_agent
# train_env = OptimalExecutionEnv(sim, T=10, seed=SEED)
# test_r = test.copy()
# test_r["regime"] = det.predict(test)
# eval_env = OptimalExecutionEnv(test_r, T=10, seed=SEED)
# model = train_agent(train_env, algorithm="PPO", total_timesteps=50_000,
#                     save_path=str(ROOT / "models"), log_path=str(ROOT / "logs"), seed=SEED)
# evaluate_agent(model, eval_env, n_episodes=100, seed=SEED)

## LLM explainability

Show 3–4 cached or live explanations for different regimes (set `ANTHROPIC_API_KEY` for live calls).

In [14]:
from src.llm_explainer import explain_execution

for regime in (0, 1):
    txt = explain_execution(
        regime=regime,
        regime_name=regime_display_name(regime, 2),
        inventory_remaining=0.6,
        action_taken=0.25,
        execution_cost_bps=4.5,
        twap_cost_bps=5.0,
        ac_cost_bps=4.8,
        sigma_t=0.018,
        liquidity_t=1.2e-5,
        use_cache=True,
    )
    print("---")
    print(txt)

---
In the calm regime (id 0), the desk executed 25.0% of the remaining book this interval with ~4.5 bps estimated cost versus TWAP ~5.0 bps and Almgren–Chriss ~4.8 bps. Daily volatility is about 1.80% with Amihud liquidity 0.0000. The pace balances market impact against inventory risk with 60.0% still to work.
---
In the elevated volatility regime (id 1), the desk executed 25.0% of the remaining book this interval with ~4.5 bps estimated cost versus TWAP ~5.0 bps and Almgren–Chriss ~4.8 bps. Daily volatility is about 1.80% with Amihud liquidity 0.0000. The pace balances market impact against inventory risk with 60.0% still to work.


## Discussion, ethics, AI use

- **Limitations:** Daily bars, simplified impact model, small ticker universe in parquet splits.
- **Ethics:** LLM explains decisions but does not place orders; human oversight remains mandatory.
- **AI use statement:** Development assistance from coding assistants; Claude API optional for governance text; design choices are the team's.

In [15]:
from src.scenario_paths import synthetic_panel
from src.trend_classifier import compute_trend_regime, TREND_DOWN, TREND_MID, TREND_UP
from src.regime_switching import TWAPFallbackPolicy, build_regime_switching_policy
from src.trading_env import OptimalExecutionEnv
from src.rl_agent import evaluate_agent, format_rl_eval_report

# Synthetic uptrend panel + trend labels (0=down, 1=mid, 2=up)
up_df = compute_trend_regime(synthetic_panel("up", n=80, step=0.3), lookback=20)
env_sw = OptimalExecutionEnv(up_df, T=10, seed=0, resample=True)
twap_pol = TWAPFallbackPolicy(env_sw)
# Trivial meta-policy: all slots TWAP (replace uptrend with your PPO .zip via scripts/train.py --regime-switch)
meta_pol = build_regime_switching_policy(twap_pol, twap_pol, env_sw, midtrend_strategy="twap")
bp_sw = {
    "T": 10,
    "X_0": 1.0,
    "eta": 0.01,
    "gamma": 0.001,
    "lam": 0.5,
    "order_notional_usd": 0.0,
    "order_start_bar": 0,
}
st_sw = evaluate_agent(meta_pol, env_sw, n_episodes=20, seed=0, bench_params=bp_sw)
print(format_rl_eval_report(st_sw))
print("selections (up/mid/down episodes):", getattr(meta_pol, "policy_selections", {}))

  mean_episode_return        0.048244
  std_episode_return         0.000000
  mean_twap_is_bps (paths)   123.2507
  mean_vwap_is_bps (paths)   123.2507
  mean_RL_minus_TWAP_bps     -0.1324  (>0 => RL higher avg sell vs arrival than TWAP on same windows)
  std_RL_minus_TWAP_bps      0.0073
  mean_RL_minus_VWAP_bps     -0.1324
  pct_beat_TWAP_IS           0.00%
  pct_beat_VWAP_IS           0.00%
  IS-gap Sharpe              -18.0380
  95% CI (TWAP gap)          [-0.1356, -0.1291]
  TWAP gap regime 0         -0.1324  (n=20)
  TWAP gap trend 1 (dn/mid/up) -0.1414  (n=6)
  TWAP gap trend 2 (dn/mid/up) -0.1285  (n=14)
selections (up/mid/down episodes): {'up': 14, 'mid': 6, 'down': 0}
